In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
import joblib
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [21]:
df=pd.read_csv('../data/healthcare_cleaned.csv')

In [22]:
drop_cols=['patient_id','treatment_cost','mortality_risk','diagnosis_category','length_of_stay','readmission_30day']

In [23]:
X=df.drop(drop_cols,axis=1)
y=df['diagnosis_category']

In [32]:
y.value_counts(normalize=True)

diagnosis_category
Cardiovascular     0.28768
Musculoskeletal    0.24050
Neurological       0.19916
Oncological        0.15170
Respiratory        0.12096
Name: proportion, dtype: float64

In [25]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [27]:
cat_cols=[col for col in X.columns if X[col].dtypes=='str']
num_cols=[col for col in X.columns if col not in cat_cols]
preprocessor=ColumnTransformer(transformers=[
    ('num',StandardScaler(),num_cols),
    ('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)
])
smote_dt=ImbPipeline(steps=[
    ('preprocessor',preprocessor),
    ('smote',SMOTE(random_state=42)),
    ('model',DecisionTreeClassifier(random_state=42))
])
smote_dt.fit(X_train,y_train)
y_pred=smote_dt.predict(X_test)
print(classification_report(y_test,y_pred))

                 precision    recall  f1-score   support

 Cardiovascular       0.30      0.30      0.30      2877
Musculoskeletal       0.26      0.24      0.25      2405
   Neurological       0.22      0.22      0.22      1991
    Oncological       0.18      0.19      0.18      1517
    Respiratory       0.16      0.17      0.17      1210

       accuracy                           0.24     10000
      macro avg       0.22      0.22      0.22     10000
   weighted avg       0.24      0.24      0.24     10000



In [30]:
from sklearn.model_selection import GridSearchCV
param_dt={
    'model__max_depth':[3,5,7],
    'model__min_samples_split':[3,5,10]
}
grid_model_dt=GridSearchCV(estimator=smote_dt,param_grid=param_dt,cv=5,scoring='f1_macro',n_jobs=-1)
grid_model_dt.fit(X_train,y_train)
y_pred=grid_model_dt.predict(X_test)
print(classification_report(y_test,y_pred))

                 precision    recall  f1-score   support

 Cardiovascular       0.41      0.36      0.38      2877
Musculoskeletal       0.37      0.31      0.34      2405
   Neurological       0.34      0.30      0.32      1991
    Oncological       0.29      0.28      0.28      1517
    Respiratory       0.24      0.44      0.31      1210

       accuracy                           0.33     10000
      macro avg       0.33      0.34      0.33     10000
   weighted avg       0.35      0.33      0.34     10000



In [31]:
print(grid_model_dt.best_params_)
print(grid_model_dt.best_score_)

{'model__max_depth': 5, 'model__min_samples_split': 3}
0.3340015987043211


In [33]:
from sklearn.pipeline import Pipeline
pipeline_rf=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',RandomForestClassifier(random_state=42,class_weight='balanced'))
])
pipeline_rf.fit(X_train,y_train)
y_pred=pipeline_rf.predict(X_test)
print(classification_report(y_test,y_pred))
param_rf={
    'model__max_depth':[3,5,7],
    'model__n_estimators':[50,100,150],
    'model__min_samples_split':[3,5,7]
}
grid_model_rf=GridSearchCV(estimator=pipeline_rf,param_grid=param_rf,cv=5,scoring='f1_macro',n_jobs=-1)
grid_model_rf.fit(X_train,y_train)
y_pred=grid_model_rf.predict(X_test)
print(classification_report(y_test,y_pred))

                 precision    recall  f1-score   support

 Cardiovascular       0.34      0.51      0.41      2877
Musculoskeletal       0.33      0.34      0.33      2405
   Neurological       0.32      0.25      0.28      1991
    Oncological       0.28      0.16      0.20      1517
    Respiratory       0.25      0.18      0.21      1210

       accuracy                           0.32     10000
      macro avg       0.30      0.29      0.29     10000
   weighted avg       0.31      0.32      0.31     10000

                 precision    recall  f1-score   support

 Cardiovascular       0.41      0.36      0.38      2877
Musculoskeletal       0.37      0.31      0.34      2405
   Neurological       0.34      0.30      0.32      1991
    Oncological       0.29      0.28      0.28      1517
    Respiratory       0.24      0.44      0.31      1210

       accuracy                           0.33     10000
      macro avg       0.33      0.34      0.33     10000
   weighted avg       0.35

In [34]:
smote_rf=ImbPipeline(steps=[
    ('preprocessor',preprocessor),
    ('smote',SMOTE(random_state=42)),
    ('model',RandomForestClassifier(random_state=42))
])
smote_rf.fit(X_train,y_train)
y_pred=smote_rf.predict(X_test)
print(classification_report(y_test,y_pred))
grid_model_rf=GridSearchCV(estimator=smote_rf,param_grid=param_rf,cv=5,scoring='f1_macro',n_jobs=-1)
grid_model_rf.fit(X_train,y_train)
y_pred=grid_model_rf.predict(X_test)
print(classification_report(y_test,y_pred))

                 precision    recall  f1-score   support

 Cardiovascular       0.36      0.46      0.40      2877
Musculoskeletal       0.33      0.32      0.33      2405
   Neurological       0.32      0.26      0.28      1991
    Oncological       0.28      0.19      0.23      1517
    Respiratory       0.25      0.27      0.26      1210

       accuracy                           0.32     10000
      macro avg       0.31      0.30      0.30     10000
   weighted avg       0.32      0.32      0.32     10000

                 precision    recall  f1-score   support

 Cardiovascular       0.41      0.36      0.38      2877
Musculoskeletal       0.37      0.31      0.34      2405
   Neurological       0.34      0.30      0.32      1991
    Oncological       0.29      0.28      0.28      1517
    Respiratory       0.24      0.44      0.31      1210

       accuracy                           0.33     10000
      macro avg       0.33      0.34      0.33     10000
   weighted avg       0.35

In [38]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
pipeline_xgb=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('model',XGBClassifier(random_state=42))
])
pipeline_xgb.fit(X_train,y_train_encoded)
y_pred=pipeline_xgb.predict(X_test)
print(classification_report(y_test_encoded,y_pred))
param_xgb={
    'model__max_depth':[3,5,7],
    'model__n_estimators':[50,100,150],
    'model__learning_rate':[0.01,0.05,0.1,0.3]
}
grid_model_xgb=GridSearchCV(estimator=pipeline_xgb,param_grid=param_xgb,cv=5,scoring='f1_macro',n_jobs=-1)
grid_model_xgb.fit(X_train,y_train_encoded)
y_pred=grid_model_xgb.predict(X_test)
print(classification_report(y_test_encoded,y_pred))

              precision    recall  f1-score   support

           0       0.34      0.49      0.40      2877
           1       0.32      0.33      0.32      2405
           2       0.29      0.23      0.26      1991
           3       0.28      0.16      0.20      1517
           4       0.23      0.18      0.20      1210

    accuracy                           0.31     10000
   macro avg       0.29      0.28      0.28     10000
weighted avg       0.30      0.31      0.30     10000

              precision    recall  f1-score   support

           0       0.37      0.44      0.40      2877
           1       0.37      0.31      0.34      2405
           2       0.34      0.30      0.32      1991
           3       0.28      0.25      0.27      1517
           4       0.24      0.28      0.26      1210

    accuracy                           0.33     10000
   macro avg       0.32      0.32      0.32     10000
weighted avg       0.33      0.33      0.33     10000



In [39]:
print(grid_model_xgb.best_params_)
print(grid_model_xgb.best_score_)

{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 150}
0.3231146219474509


Observations:
1) All three models (DT, RF, XGBoost) hit same ceiling ~0.33 macro F1
2) Respiratory and Oncological consistently hardest — smallest classes
3) Cardiovascular easiest — largest class, most training samples
4) Weak feature signal root cause — same conclusion as previous notebooks
5) Best model: RF + class_weight, macro F1=0.33

In [40]:
joblib.dump(grid_model_rf, '../models/diagnosis_model.pkl')

['../models/diagnosis_model.pkl']